In [4]:
import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta

# --- File paths ---
IN_PATH  = "swaps.xlsx"
OUT_CSV  = "discounts.csv"

# --- Day count / date helpers ---
def act_360(start, end):
    return (end - start).days / 360.0

def add_months(d, months):
    return d + relativedelta(months=int(months))

def add_years(d, years):
    return d + relativedelta(years=int(years))

# --- Load quotes ---
data = pd.read_excel(IN_PATH, sheet_name="swaps")
data["date"] = pd.to_datetime(data["date"])

# ----------------------------
# OUTPUT GRID: 1d + monthly nodes up to 10y, columns in YEAR FRACTIONS
# ----------------------------
MAX_YEARS = 10
months_grid = list(range(1, MAX_YEARS * 12 + 1))  # 1..120 months
tenors_yearfrac = [1.0/360.0] + [m/12.0 for m in months_grid]  # year fractions

def tenor_label(yf):
    return f"{yf:.6f}"   # readable year-fraction label

rows_out = []

def df_from_known(curve_date, DF_map, target_date):
    """
    DF(target_date) using linear interpolation on ZERO RATES z(t) = -ln(DF)/t.
    This avoids piecewise-constant forwards that you get with log-linear DF.
    Extrapolation: keeps your previous piecewise-constant-forward approach.
    """
    known = sorted(DF_map.keys())
    if not known:
        return np.nan

    tt = act_360(curve_date, target_date)

    earlier = [d for d in known if d <= target_date]
    later   = [d for d in known if d >= target_date]

    # ---------- INTERPOLATION (inside anchors) ----------
    if earlier and later:
        d0 = max(earlier); d1 = min(later)
        if d0 == d1:
            return DF_map[d0]

        t0 = act_360(curve_date, d0)
        t1 = act_360(curve_date, d1)
        df0 = DF_map[d0]
        df1 = DF_map[d1]

        if t1 == t0 or df0 <= 0 or df1 <= 0 or t0 <= 0 or t1 <= 0:
            # fall back to log-linear DF if times are degenerate
            w = 0.0 if t1 == t0 else (tt - t0) / (t1 - t0)
            return np.exp((1-w)*np.log(df0) + w*np.log(df1))

        z0 = -np.log(df0) / t0
        z1 = -np.log(df1) / t1
        w  = (tt - t0) / (t1 - t0)
        zt = (1-w)*z0 + w*z1
        return np.exp(-zt * tt)

    # ---------- RIGHT EXTRAPOLATION (keep your old approach) ----------
    elif earlier:
        if len(earlier) >= 2:
            d0 = earlier[-2]; d1 = earlier[-1]
            t0 = act_360(curve_date, d0); t1 = act_360(curve_date, d1)
            df0 = DF_map[d0]; df1 = DF_map[d1]
            if t1 == t0 or df0 <= 0 or df1 <= 0:
                return DF_map[d1]
            f = -np.log(df1/df0) / (t1 - t0)
            return DF_map[d1] * np.exp(-f * (tt - t1))
        else:
            return DF_map[earlier[-1]]

    # ---------- LEFT EXTRAPOLATION (keep your old approach) ----------
    elif later:
        if len(later) >= 2:
            d0 = later[0]; d1 = later[1]
            t0 = act_360(curve_date, d0); t1 = act_360(curve_date, d1)
            df0 = DF_map[d0]; df1 = DF_map[d1]
            if t1 == t0 or df0 <= 0 or df1 <= 0:
                return DF_map[d0]
            f = -np.log(df1/df0) / (t1 - t0)
            return DF_map[d0] * np.exp(f * (tt - t0))
        else:
            return DF_map[later[0]]

    return np.nan

for _, row in data.iterrows():
    curve_date = pd.Timestamp(row["date"]).normalize()

    needed = ["1d","90d","180d","1y","2y"]
    if any(pd.isna(row[k]) for k in needed):
        out = {"date": curve_date}
        for yf in tenors_yearfrac:
            out[tenor_label(yf)] = np.nan
        rows_out.append(out)
        continue

    DF = {}

    # --- Zero-coupon region (simple ACT/360) ---
    r_1d = row["1d"]/100.0
    d_1d = curve_date + pd.Timedelta(days=1)
    DF[d_1d] = 1.0 / (1.0 + r_1d * act_360(curve_date, d_1d))

    r_90 = row["90d"]/100.0
    d_90 = curve_date + pd.Timedelta(days=90)
    DF[d_90] = 1.0 / (1.0 + r_90 * act_360(curve_date, d_90))

    r_180 = row["180d"]/100.0
    d_180 = curve_date + pd.Timedelta(days=180)
    DF[d_180] = 1.0 / (1.0 + r_180 * act_360(curve_date, d_180))

    r_1y = row["1y"]/100.0
    d_1y = add_years(curve_date, 1)
    DF[d_1y] = 1.0 / (1.0 + r_1y * act_360(curve_date, d_1y))

    # Extend to 1.5y using 1y simple rate
    d_1p5y = add_months(curve_date, 18)
    DF[d_1p5y] = DF[d_1y] / (1.0 + r_1y * act_360(d_1y, d_1p5y))

    # --- Swap bootstrap (semiannual fixed leg, ACT/360) ---
    def bootstrap_swap(maturity_years, par_rate_annual):
        if pd.isna(par_rate_annual):
            return
        n = int(round(maturity_years * 2))
        coupon_dates = [add_months(curve_date, 6*i) for i in range(1, n+1)]

        alphas = []
        prev = curve_date
        for dt in coupon_dates:
            alphas.append(act_360(prev, dt))
            prev = dt

        S = par_rate_annual / 100.0

        A = 0.0
        for i in range(n-1):
            ti = coupon_dates[i]
            dfi = DF.get(ti, df_from_known(curve_date, DF, ti))
            A += alphas[i] * dfi

        DF_Tn = (1.0 - S * A) / (1.0 + S * alphas[-1])
        DF[coupon_dates[-1]] = DF_Tn

    for col in ["2y","3y","4y","5y","10y"]:
        bootstrap_swap(int(col.rstrip("y")), row[col])

    # --- Collect outputs: 1d + monthly grid (as year fractions) ---
    out = {"date": curve_date}

    # 1d tenor
    out[tenor_label(1.0/360.0)] = DF[d_1d]

    # monthly tenors
    for m in months_grid:
        yf = m/12.0
        target = add_months(curve_date, m)
        out[tenor_label(yf)] = df_from_known(curve_date, DF, target)

    rows_out.append(out)

df_out = pd.DataFrame(rows_out).sort_values("date").reset_index(drop=True)
df_out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")

Saved: discounts.csv
